# HP-Grid-Search (ML4SCS)
Config-getriebene Grid-Search der Deep-Modelle auf grouped-5-fold.
Protokoll: docs/superpowers/specs/2026-07-03-colab-grid-search-design.md.

**Vorbereitung (einmalig):** In Colab links unter 🔑 *Secrets* anlegen und fuer dieses Notebook freigeben: `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_ENDPOINT`. Runtime: GPU (T4) empfohlen.

Resumierbar: fertige Trials liegen auf R2 und werden uebersprungen — ein Disconnect verliert hoechstens den laufenden Trial.

**RunPod (Alternative zu Colab):** Pod mit Template "RunPod PyTorch" deployen, beim Deploy unter *Environment Variables* dieselben drei Namen als Env-Vars setzen (`R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_ENDPOINT`), Container-Disk ~30 GB. Dann JupyterLab oeffnen -> Terminal -> Repo klonen -> dieses Notebook ausfuehren. Die Secrets-Zelle faellt automatisch auf Env-Vars zurueck; alles Weitere (rclone, R2-Resume, Animation) ist identisch.

In [ ]:
import os, sys, subprocess
if not os.path.isdir('ML4SCS_Burk_macht_Bock'):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'development',
        'https://github.com/noahsa16/ML4SCS_Burk_macht_Bock.git'], check=True)
%cd ML4SCS_Burk_macht_Bock
%pip install -q -r requirements.txt ipywidgets
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU-only')

In [ ]:
def _secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    v = os.environ.get(name)
    if not v:
        raise SystemExit(
            f'Secret {name} fehlt — in Colab unter dem Schluessel-Symbol anlegen, '
            f'auf RunPod als Pod-Environment-Variable setzen.')
    return v

os.environ['RCLONE_CONFIG_R2_TYPE'] = 's3'
os.environ['RCLONE_CONFIG_R2_PROVIDER'] = 'Cloudflare'
os.environ['RCLONE_CONFIG_R2_ACCESS_KEY_ID'] = _secret('R2_ACCESS_KEY_ID')
os.environ['RCLONE_CONFIG_R2_SECRET_ACCESS_KEY'] = _secret('R2_SECRET_ACCESS_KEY')
os.environ['RCLONE_CONFIG_R2_ENDPOINT'] = _secret('R2_ENDPOINT')
!curl -fsSL https://rclone.org/install.sh | bash > /dev/null 2>&1 || which rclone
!rclone version | head -1

In [ ]:
# Probanden-Bundle von R2 + Integritaets-Check + Resume-Stand herunterladen
!rclone copy r2:ml4scs-sweep/sweep_data.zip . --s3-no-check-bucket
!unzip -tq sweep_data.zip && unzip -oq sweep_data.zip
!rclone copy r2:ml4scs-sweep/hp_grid models/hp_grid --s3-no-check-bucket
!find models/hp_grid -name 'trial_*.csv' 2>/dev/null | wc -l

In [ ]:
RUN = ['tcn6', 'tcn6w32', 'tcn6se', 'tcn6ap', 'tcn6k5', 'tcn6wn', 'tcn8']
# ^ Config-Stems aus configs/hp/ — fuer einen Smoke: RUN = ['smoke_tcn']

In [ ]:
import collections
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import clear_output
from tqdm.notebook import tqdm as ntqdm
sys.path.insert(0, os.getcwd())
from src.training import events as EV

class LiveView:
    '''Event-Sink: 3 Progress-Bars (Folds/Epochen) + Lernkurve + Leaderboard.'''
    def __init__(self, outdir_fn, max_epochs):
        self.outdir_fn, self.max_epochs = outdir_fn, max_epochs
        self.trial = ''
        self.fold_bar = self.epoch_bar = None
        self.curves = collections.defaultdict(list)
    def set_trial(self, name):
        self.trial = name
        self.curves.clear()
    def __call__(self, ev):
        t = ev.get('type')
        if t == EV.RUN_START:
            self.curves.clear()
            if self.fold_bar: self.fold_bar.close()
            self.fold_bar = ntqdm(total=ev['n_folds'], desc=f'{self.trial} folds', leave=False)
        elif t == EV.FOLD_START:
            self.fold = ev['person']
            if self.epoch_bar: self.epoch_bar.close()
            self.epoch_bar = ntqdm(total=self.max_epochs, desc=f'fold {self.fold}', leave=False)
        elif t == EV.EPOCH:
            self.epoch_bar.update(1)
            self.curves[self.fold].append((ev['epoch'], ev['loss'], ev['val_loss'], ev['val_auc']))
            if ev['epoch'] % 2 == 0: self.draw()
        elif t == EV.FOLD_END:
            if self.epoch_bar: self.epoch_bar.close()
            self.fold_bar.update(1)
    def draw(self):
        clear_output(wait=True)
        fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 4))
        pts = self.curves[self.fold]
        if pts:
            e, tl, vl, va = zip(*pts)
            ax.plot(e, tl, label='train loss'); ax.plot(e, vl, label='val loss')
            axa = ax.twinx(); axa.plot(e, va, color='g', alpha=0.5, label='val AUC')
            axa.set_ylim(0.5, 1.0)
            ax.set_title(f'{self.trial} — fold {self.fold} (Epoche {e[-1]})')
            ax.set_xlabel('Epoche'); ax.legend(loc='upper right')
        rows = []
        for f in self.outdir_fn().glob('trial_*.csv'):
            rows.append(pd.read_csv(f))
        if rows:
            lb = (pd.concat(rows).groupby('cfg_id')['accuracy'].mean()
                  .sort_values().tail(12))
            colors = ['#2a9d8f' if i == len(lb)-1 else '#bbb' for i in range(len(lb))]
            ax2.barh(lb.index, lb.values, color=colors)
            ax2.set_xlim(max(0.5, lb.min()-0.02), min(1.0, lb.max()+0.01))
            ax2.set_title(f'Leaderboard (Seed-Mittel-Acc, {lb.index.nunique()} Configs fertig)')
        plt.tight_layout(); plt.show()

In [ ]:
from pathlib import Path
from src.training.deep.grid import run_grid, collect_grid, grid_outdir, load_grid_spec

def sync_up(outdir):
    subprocess.run(['rclone', 'copy', str(outdir),
                    f'r2:ml4scs-sweep/hp_grid/{outdir.parent.name}/{outdir.name}',
                    '--s3-no-check-bucket'], check=True)

for stem in ntqdm(RUN, desc='Modelle'):
    cfg = Path('configs/hp') / f'{stem}.json'
    spec = load_grid_spec(cfg)
    view = LiveView(lambda c=cfg, s=spec: grid_outdir(s, c), spec.max_epochs)
    from src.training.deep.hp_search import grid_configs
    n = len(grid_configs(spec.grid.model_dump())) * len(spec.seeds)
    print(f'== {stem}: {n} Trials (Configs x Seeds), Resume aktiv ==')
    view.set_trial(stem)
    run_grid(cfg, on_event=view, after_trial=sync_up)
    sync_up(grid_outdir(spec, cfg))

In [ ]:
for stem in RUN:
    collect_grid(Path('configs/hp') / f'{stem}.json')
!rclone copy models r2:ml4scs-sweep/hp_grid/_collect --include 'grid_*csv' --s3-no-check-bucket

## Replay nach Disconnect
Lernkurven aus den gespeicherten History-CSVs — kein Neu-Training noetig.

In [ ]:
stem = RUN[0]
spec = load_grid_spec(Path('configs/hp') / f'{stem}.json')
outdir = grid_outdir(spec, Path('configs/hp') / f'{stem}.json')
hist_files = sorted(outdir.glob('history_*.csv'))
print(f'{len(hist_files)} History-Dateien')
if hist_files:
    h = pd.read_csv(hist_files[0])
    for held_out, g in h.groupby('held_out'):
        plt.plot(g['epoch'], g['val_auc'], alpha=0.6, label=str(held_out)[:18])
    plt.xlabel('Epoche'); plt.ylabel('val AUC'); plt.legend(fontsize=7)
    plt.title(hist_files[0].stem); plt.show()

In [ ]:
# Fallback-Download als Zip (falls R2 nicht erreichbar)
!zip -qr hp_grid_results.zip models/hp_grid models/grid_*csv 2>/dev/null || true
try:
    from google.colab import files
    files.download('hp_grid_results.zip')
except ImportError:
    print('Nicht-Colab: hp_grid_results.zip liegt im Arbeitsverzeichnis '
          '(JupyterLab-Dateibrowser) — Ergebnisse sind ohnehin auf R2.')